# EmbeddingGemma 2 日英混合文字起こし — 精度改善版

従来のノートブックでは **日本語500件・英語500件 / 5エポック / 語彙6000件** を標準とし、
検証時に先頭30件（実質日本語だけ）で最良モデルを決定する問題がありました。

改善版では (1) 日英別・混合別のCERを独立測定しmacro平均で最良を選択、
(2) 英語の学習済みReaderを教師として英語の表現を維持、(3) cosine LR/early stopping、
(4) 日英音声を連結した補助的な混合学習データ、(5) FLEURSの全サンプルを使える初期値を追加。

重要: 日英連結による合成音声は「発話境界での切り替え」を練習するものです。
自然な文中切り替えを実現するには、**自然な日英混合音声と正解テキストを追加**してください。
音声エンコーダーと既存en.eg2mは変更しません。T4 GPU (FP32) をサポートします。

ノートブックを初めて実行するなら①→⑫を実行。
**以前のデータ・語彙・表を維持して⑩のみ改善**する場合は、③→⑩→⑪→⑫を実行
（①と⑨を以前のランタイムで実行済みであること）。学習済みReaderの継続は
同じ語彙・同じword_table.npyの場合に限定してください。


In [ ]:
# @title ① 設定（精度改善・英日混合）
VOCAB_MODE = 'word'  # @param ['word','character']
MAX_VOCAB = 4000  # @param {type:'integer'}
TRAIN_LIMIT_PER_LANGUAGE = 0  # @param {type:'integer'}
DEV_LIMIT_PER_LANGUAGE = 0  # @param {type:'integer'}
# 0 = FLEURSの対象splitを全部使用。必要なら500などで試験できます。
ADD_SYNTHETIC_MIX = True  # @param {type:'boolean'}
TRAIN_SYNTHETIC_PAIRS = 500  # @param {type:'integer'}
DEV_SYNTHETIC_PAIRS = 80  # @param {type:'integer'}
# 実際の日英混合音声JSONLを追加可能。各行 {"id":...,"audio":... ,"text":...}
EXTRA_TRAIN_MANIFEST = ''  # @param {type:'string'}
EXTRA_DEV_MANIFEST = ''  # @param {type:'string'}
EPOCHS = 14  # @param {type:'integer'}
BATCH_SIZE = 2  # @param {type:'integer'}
LEARNING_RATE = 1.2e-5  # @param {type:'number'}
WARMUP_STEPS = 100  # @param {type:'integer'}
ENGLISH_DISTILL_WEIGHT = 0.5  # @param {type:'number'}
EARLY_STOP_PATIENCE = 4  # @param {type:'integer'}
MAX_EVAL_PER_LANGUAGE = 70  # @param {type:'integer'}
DEVICE_PREFERENCE = 'auto'  # @param ['auto','cuda','cpu']
WORD_TABLE_BATCH = 16  # @param {type:'integer'}
EMBEDDING_DTYPE = 'auto'  # @param ['auto','fp32','bf16']
INITIALIZE_FROM_ENGLISH = True  # @param {type:'boolean'}
# 語彙・単語表を一切変更しない時のみ、既存Readerの.ptを指定して継続学習。
RESUME_READER_PT = ''  # @param {type:'string'}
USE_FLEURS = True  # @param {type:'boolean'}
TRAIN_MANIFEST = '/content/mixed_train.jsonl'  # @param {type:'string'}
DEV_MANIFEST = '/content/mixed_dev.jsonl'  # @param {type:'string'}
SAVE_TO_DRIVE = False  # @param {type:'boolean'}
BROWSER_DOWNLOAD = True  # @param {type:'boolean'}

from pathlib import Path
BASE=Path('/content/eg2_mix')
TOOLS=BASE/'tools'; MODEL=BASE/'source_encoder'; DATA=BASE/'dataset'
VOCAB=BASE/'reader_vocab.txt'; TABLE=BASE/'word_table.npy'
CACHE_TRAIN=BASE/'cache/train'; CACHE_DEV=BASE/'cache/dev'
TRAINED=BASE/'trained'; READY=BASE/'models-mix'; OUTPUT=BASE/'embeddinggemma2-reader-en-ja-quality.zip'
assert VOCAB_MODE in ('word','character') and MAX_VOCAB>0
import torch
DEVICE = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE_PREFERENCE == 'auto' else DEVICE_PREFERENCE
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('ColabランタイムをT4 GPU等に変更してください')
if DEVICE == 'cuda':
    props=torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} / {props.total_memory/2**30:.1f} GiB / BF16={torch.cuda.is_bf16_supported()}')
else:print('CPU実行（遅くなります）')
print('Device:',DEVICE)


In [ ]:
# @title ② Python依存をインストール（実行時のみ）
import sys,subprocess
subprocess.run([sys.executable,'-m','pip','install','-q',
                'transformers>=5.19.0','sentence-transformers>=6.1.0',
                'huggingface_hub','datasets','soundfile','scipy',
                'sudachipy','sudachidict_core','torchcodec'],check=True)
for directory in (TOOLS,MODEL,DATA,CACHE_TRAIN,CACHE_DEV,TRAINED,READY):directory.mkdir(parents=True,exist_ok=True)
print('依存ライブラリ準備完了')

In [ ]:
# @title ③ 改善版の学習ツールを展開（別のソースアップロード不要）
import base64,io,zipfile
BUNDLE = """..."""
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    expected={'eg2_ja_vocab.py','eg2_mix_vocab.py','eg2_ja_table.py','eg2_ja_cache.py','eg2_ja_train.py','eg2_ja_fleurs.py',
              'eg2_mix_augment.py','convert_eg2_reader.py','convert_eg2_table.py','eg2_bundle.py'}
    assert set(z.namelist())==expected
    z.extractall(TOOLS)
print('ツール:',len(expected))


## ④ 学習データについて

`USE_FLEURS=True`で日本語・英語FLEURSの学習splitとvalidation splitを使用します。
`TRAIN_LIMIT_PER_LANGUAGE=0`は利用可能な全件を意味します。
別々の日本語/英語WAVを連結した合成データも追加できますが、文中の自然なコードスイッチは別途録音して
`EXTRA_TRAIN_MANIFEST` / `EXTRA_DEV_MANIFEST` に追加してください。
音声は16kHz・モノラルWAV、30秒以下を推奨。train/devで音声や話者を重複させないでください。


In [ ]:
# @title ④ 日英音声を用意 + 発話境界での混合音声を追加（実音声混合にも対応）
import json, subprocess,sys
if USE_FLEURS:
    for language in ('ja_jp','en_us'):
        subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_ja_fleurs.py'),
            '--out',str(DATA/language),'--language',language,
            '--train-limit',str(TRAIN_LIMIT_PER_LANGUAGE),
            '--dev-limit',str(DEV_LIMIT_PER_LANGUAGE)],check=True)
    for split in ('train','dev'):
        target=DATA/(split+'.jsonl')
        with target.open('w',encoding='utf8') as out:
            for language in ('ja_jp','en_us'):
                for row in (DATA/language/(split+'.jsonl')).read_text(encoding='utf8').splitlines():
                    item=json.loads(row);item['id']=language+'_'+item['id'];item['language']=language
                    out.write(json.dumps(item,ensure_ascii=False)+'\n')
    TRAIN_MANIFEST=str(DATA/'train.jsonl'); DEV_MANIFEST=str(DATA/'dev.jsonl')
else:
    if ADD_SYNTHETIC_MIX:raise ValueError('Synthetic mixはFLEURSから生成します。USE_FLEURS=False時はADD_SYNTHETIC_MIX=Falseにしてください')

for split in ('train','dev'):
    manifest=Path(TRAIN_MANIFEST if split=='train' else DEV_MANIFEST)
    if not manifest.is_file():raise FileNotFoundError(manifest)
    if USE_FLEURS and ADD_SYNTHETIC_MIX:
        mixed_out=DATA/(split+'_aug.jsonl')
        subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_mix_augment.py'),
            '--input',str(manifest),'--output',str(mixed_out),
            '--pairs',str(TRAIN_SYNTHETIC_PAIRS if split=='train' else DEV_SYNTHETIC_PAIRS),
            '--seed',str(19 if split=='train' else 53)],check=True)
        manifest=mixed_out
    extra=EXTRA_TRAIN_MANIFEST if split=='train' else EXTRA_DEV_MANIFEST
    if extra:
        extra=Path(extra)
        if not extra.is_file():raise FileNotFoundError(extra)
        merged=DATA/(split+'_with_real_mix.jsonl')
        with merged.open('w',encoding='utf8') as out:
            for line in manifest.read_text(encoding='utf8').splitlines():
                if line.strip():out.write(line+'\n')
            for i,line in enumerate(extra.read_text(encoding='utf8').splitlines()):
                if not line.strip():continue
                item=json.loads(line)
                a=Path(item['audio'])
                if not a.is_absolute():a=extra.resolve().parent/a
                item['audio']=str(a.resolve());item['id']=f'real_mix_{split}_{i:06d}'
                out.write(json.dumps(item,ensure_ascii=False)+'\n')
        manifest=merged
    if split=='train':TRAIN_MANIFEST=str(manifest)
    else:DEV_MANIFEST=str(manifest)
    print(split,'examples:',len(manifest.read_text(encoding='utf8').splitlines()),'path:',manifest)


必要なら Hugging Face の利用条件に同意し、認証を行ってください。下のセルでは `huggingface_hub.snapshot_download` を使い、**Encoder およびテキスト埋め込み用の公式重み**を取得します。Google のモデル利用規約・アクセス条件が適用されます。

In [ ]:
# @title 任意：Hugging Face 認証（アクセスが拒否された場合だけ、このセルを手動で実行）
# from huggingface_hub import login
# login()
print('認証不要ならこのまま次へ進みます。')

In [ ]:
# @title ⑤ EmbeddingGemma 2 公式モデルを取得
from huggingface_hub import snapshot_download,hf_hub_download
snapshot_download(repo_id='google/embeddinggemma-2',local_dir=str(MODEL),
    allow_patterns=['*.json','*.safetensors','*.model','tokenizer*','merges.txt','vocab.*'])
print('encoder / embedding model:',MODEL)

In [ ]:
# @title ⑥ 日英共通語彙を作り直す（語彙変更時は必ず⑦も再実行）
subprocess.run([sys.executable,str(TOOLS/'eg2_mix_vocab.py'),
    '--train',str(TRAIN_MANIFEST),'--mode',VOCAB_MODE,
    '--max-vocab',str(MAX_VOCAB),'--min-count','3',
    '--out',str(VOCAB)],check=True)
print('共同語彙:',len(VOCAB.read_text(encoding='utf8').splitlines()))


In [ ]:
# @title ⑦ 日英共通単語の768次元ベクトルを生成（実際のEmbeddingGemma 2を使用）
subprocess.run([sys.executable,'-u',str(TOOLS/'eg2_ja_table.py'),
    '--model-dir',str(MODEL),'--vocab',str(VOCAB),'--out',str(TABLE),
    '--device',DEVICE,'--dtype',EMBEDDING_DTYPE,'--batch',str(WORD_TABLE_BATCH)],check=True)
print('word_table.npy 完成:',TABLE)

In [ ]:
# @title ⑧ 日英の音声 → 512次元 Soft Tokens のキャッシュ（詳細ログ付き）
# 失敗時は最後のCalledProcessErrorだけでなく、内部Python tracebackを画面とログに保存。
# 既にできた .npy は再利用するので、失敗後の再実行で最初から計算し直しません。
import subprocess,sys,torch

# Select the actual device *now*. The runtime/GPU may have changed since ①.
cuda_now = bool(torch.cuda.is_available())
if DEVICE_PREFERENCE == 'auto':
    DEVICE = 'cuda' if cuda_now else 'cpu'
if DEVICE == 'cuda' and not cuda_now:
    raise RuntimeError('CUDAが利用できません。ColabのランタイムをT4 GPUに切り替えて①と⑧を実行してください。')
if DEVICE == 'cuda':
    print('⑧ CUDA GPU:',torch.cuda.get_device_name(0), ' dtype:',EMBEDDING_DTYPE)
else:
    print('⑧ CPUで実行中 (T4を使う場合: ランタイム→ランタイムのタイプを変更→T4 GPU)。',flush=True)

def run_cache_logged(split, manifest, out_dir):
    logfile=BASE/('eg2_cache_'+split+'.log')
    command=[sys.executable,'-u',str(TOOLS/'eg2_ja_cache.py'),
        '--manifest',str(manifest),'--model-dir',str(MODEL),
        '--out',str(out_dir),'--device',DEVICE,'--dtype',EMBEDDING_DTYPE]
    print('>>>',split,manifest,' | log:',logfile,flush=True)
    with logfile.open('w',encoding='utf-8') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,encoding='utf-8',errors='replace',bufsize=1) as proc:
            for line in proc.stdout:
                print(line,end='',flush=True)
                log.write(line)
            returncode=proc.wait()
    if returncode:
        print('失敗時の詳細ログ:',logfile)
        print('Colabで確認: !tail -n 100 '+str(logfile))
        raise RuntimeError(f'{split} 音声キャッシュ失敗 (code={returncode})。上のtracebackを参照してください。')

for split,manifest,out_dir in [('train',TRAIN_MANIFEST,CACHE_TRAIN),('dev',DEV_MANIFEST,CACHE_DEV)]:
    run_cache_logged(split,manifest,out_dir)
print('音声特徴のキャッシュを生成しました')

In [ ]:
# @title ⑨ 既存の英語 Reader v2 を初期値として取得（日英共通Readerに追加学習する）
INIT=[]
if INITIALIZE_FROM_ENGLISH:
    pretrained=hf_hub_download(repo_id='mlboydaisuke/EmbeddingGemma-2-Transcribe-Reader',filename='reader.pt')
    INIT=['--init-reader',str(pretrained)]
    print('英語Readerを初期値にします。日本語辞書のみの差し替えではありません。')
else:
    print('ランダム初期化から学習します。より多くの音声・計算が必要です。')

In [ ]:
# @title ⑩ 品質改善CTC学習（言語別CER/英語Reader保持/early stopping）
import subprocess,sys,torch,json,hashlib
if not (CACHE_TRAIN/'index.jsonl').is_file() or not (CACHE_DEV/'index.jsonl').is_file():
    raise FileNotFoundError('⑧でtrain/dev音声キャッシュを生成してください')
if not VOCAB.is_file() or not TABLE.is_file():
    raise FileNotFoundError('⑥と⑦を実行してください')
if DEVICE_PREFERENCE=='auto':DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE.startswith('cuda') and not torch.cuda.is_available():raise RuntimeError('CUDA利用不可')
print('⑩ Device:',DEVICE,torch.cuda.get_device_name(0) if DEVICE.startswith('cuda') else '')

RESUME_ARGS=[]
if RESUME_READER_PT:
    resume=Path(RESUME_READER_PT)
    if not resume.is_file():raise FileNotFoundError(resume)
    meta=resume.parent/'training_info.json'
    if not meta.is_file():raise ValueError('再開には以前のtraining_info.jsonが必要です。語彙と単語表の一致を確認できません。')
    old=json.loads(meta.read_text(encoding='utf8'))
    for path,key in [(VOCAB,'vocab_sha256'),(TABLE,'table_sha256')]:
        digest=hashlib.sha256(path.read_bytes()).hexdigest()
        if key not in old or old[key]!=digest:
            raise ValueError(f'既存Readerは異なる語彙または表で学習されています: {key}。再開しないで英語Readerから学習してください。')
    RESUME_ARGS=['--resume-reader',str(resume)]

LOGFILE=BASE/'eg2_train_ctc_quality.log'
def run_train(batch_size):
    command=[sys.executable,'-u',str(TOOLS/'eg2_ja_train.py'),
        '--train',str(CACHE_TRAIN/'index.jsonl'),'--dev',str(CACHE_DEV/'index.jsonl'),
        '--vocab',str(VOCAB),'--table',str(TABLE),'--out',str(TRAINED),
        '--mixed','--layers','6','--epochs',str(EPOCHS),'--batch',str(batch_size),
        '--max-eval',str(MAX_EVAL_PER_LANGUAGE),'--device',DEVICE,
        '--lr',str(LEARNING_RATE),'--warmup-steps',str(WARMUP_STEPS),
        '--english-distill',str(ENGLISH_DISTILL_WEIGHT if INITIALIZE_FROM_ENGLISH else 0),
        '--patience',str(EARLY_STOP_PATIENCE),*INIT,*RESUME_ARGS]
    print('CTC train batch=',batch_size,'log=',LOGFILE,flush=True)
    with LOGFILE.open('a',encoding='utf8') as log:
        log.write('\n===== new quality training run batch='+str(batch_size)+' =====\n')
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,encoding='utf8',errors='replace',bufsize=1) as proc:
            recent=[]
            for line in proc.stdout:
                print(line,end='',flush=True);log.write(line);log.flush()
                recent.append(line)
                if len(recent)>140:recent.pop(0)
            code=proc.wait()
    return code,''.join(recent)

status,output=run_train(BATCH_SIZE)
if status and BATCH_SIZE>1 and 'out of memory' in output.lower():
    if torch.cuda.is_available():torch.cuda.empty_cache()
    print('CUDA OOM: batch=1 で再試行',flush=True)
    status,output=run_train(1)
if status:
    raise RuntimeError('⑩学習失敗。'+str(LOGFILE)+' に記録された traceback を確認してください')
print('学習完了。最良のReader:',TRAINED/'reader_ja.pt')
print('日英別CER:',TRAINED/'history.json')


学習中、エポックごとに `dev_macro_CER` と `by_language={'ja':.., 'en':.., 'mixed':..}` を表示します。
**CERは低いほど良い**です。`mixed` は合成データであるため、実際の日英混合音声でも別途評価してください。
`!tail -n 120 /content/eg2_mix/eg2_train_ctc_quality.log` で内部例外を確認できます。
語彙や word_table.npy を作り直した場合、元の .pt をそのまま再開しないでください。


In [ ]:
# @title ⑪ ネイティブC用 Reader／INT8辞書に変換して検査
import shutil,json,struct
subprocess.run([sys.executable,str(TOOLS/'convert_eg2_reader.py'),
    str(TRAINED/'reader_ja.pt'),str(READY/'reader.eg2r')],check=True)
subprocess.run([sys.executable,str(TOOLS/'convert_eg2_table.py'),
    str(TABLE),str(READY/'word_table.eg2q')],check=True)
shutil.copyfile(VOCAB,READY/'reader_vocab.txt')
with (READY/'word_table.eg2q').open('rb') as f:
    magic,version,nwords,width=struct.unpack('<4sIII',f.read(16))
assert magic==b'EG2Q' and version==1 and width==768
assert nwords==len(VOCAB.read_text(encoding='utf8').splitlines())
assert (READY/'word_table.eg2q').stat().st_size==16+nwords*768
shutil.copyfile(TRAINED/'training_info.json',READY/'training_info.json')
print('native bilingual Reader:',nwords,'tokens')

In [ ]:
# @title ⑫ 最良Readerを mix.eg2m にまとめる（以前のモデルをバックアップ）
import zipfile,shutil
package=READY/'mix.eg2m'
if package.is_file():shutil.copy2(package,READY/'mix_previous.eg2m')
subprocess.run([sys.executable,str(TOOLS/'eg2_bundle.py'),
    '--mix-dir',str(READY),'--out',str(READY),'--force'],check=True)
with zipfile.ZipFile(OUTPUT,'w',compression=zipfile.ZIP_STORED,allowZip64=True) as z:
    z.write(package,'models/mix.eg2m')
    z.write(TRAINED/'history.json','quality/history.json')
    z.write(TRAINED/'training_info.json','quality/training_info.json')
print('出力:',OUTPUT,'MiB:',round(OUTPUT.stat().st_size/2**20,2))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    dest=Path('/content/drive/MyDrive/EmbeddingGemma2');dest.mkdir(parents=True,exist_ok=True)
    shutil.copy2(OUTPUT,dest/OUTPUT.name)
if BROWSER_DOWNLOAD:
    from google.colab import files
    files.download(str(OUTPUT))


## Cでの利用と精度確認

```bash
make
./cats -a eg2-native-asr -m models -L en   -w english.wav  # 旧英語版（比較）
./cats -a eg2-native-asr -m models -L auto -w english.wav  # 混合版
./cats -a eg2-native-asr -m models -L auto -w japanese.wav
./cats -a eg2-native-asr -m models -L auto -w bilingual.wav
```

日本語・英語・同一発話の日英混合を別々に評価してください。
同じWAVを旧英語版と日英混合版で比べると、英語の能力が保持されているか分かります。
既存の `encoder.eg2m`、`en.eg2m` はそのままで、今回の `mix.eg2m` のみ置き換えます。

`training_info.json` と `history.json` のCERは学習中の検証値であり、
未学習の実音声に対する精度を保証する数字ではありません。
